In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.optimize import minimize, lsq_linear
from scipy.special import expit
from sklearn.model_selection import KFold

from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.ticker import AutoMinorLocator, FormatStrFormatter, MaxNLocator, MultipleLocator
from matplotlib.colors import to_rgba
from matplotlib.lines import Line2D
from statsmodels.stats.multitest import multipletests


MODEL_NAME = 'SE-Hurdle-S'
MODEL_T5_NAME = 'SE-Hurdle-T5'
HURDLE_AR5_NAME = 'Hurdle-AR(5)-S-P'
MODEL_TAG = 'se_hurdle_s_t5_hurdle_ar5_s_p'

SEED = 63
EPS = 0.49
Y = 20
N_DIAGNOSTIC_REPS = 300
N_REPEATED_CV = 30
PRACTICAL_NLL_TOL = 0.01
N_GAP_BOOT = 10000


SELF_RESIDUAL_DIST = 'normal'
HURDLE_RESIDUAL_DIST = 'normal'
CONSTRAIN_EXCITATION = True
TRUNCATED_HISTORY_LENGTH = 5
AR_ORDER = 5

RHO_GRID = np.linspace(0.0, 0.98, 121)
N_SPLITS = 5
N_SIM = 10000
N_RANK_REPS = 200
N_BOOT = 1000
DPI = 300

TASK_ROOT = Path.cwd().parent
PROJ_ROOT = TASK_ROOT.parent
OUTPUT = TASK_ROOT/'output'/'t5_p_output'
RESULT_DIR = OUTPUT / 'results'
FIG_DIR = OUTPUT/'figures'
DATA = TASK_ROOT/'input'

RESULT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

COMPARISON_LABELS = ('Empirical', MODEL_NAME, MODEL_T5_NAME, HURDLE_AR5_NAME)


In [ ]:
MODEL_BANK = {}
MODEL_PRESETS = {}

def register_model(label, q):
    q = np.asarray(q)
    if q.ndim != 2:
        raise ValueError(f'{label}: expected a 2D trajectory array')
    if q.shape[1] != Y + 1 and q.shape[0] == Y + 1:
        q = q.T
    if q.shape[1] != Y + 1:
        raise ValueError(f'{label}: expected {Y + 1} years, found shape {q.shape}')

    MODEL_BANK[label] = q
    return q

def register_preset(name, *labels):
    MODEL_PRESETS[name] = labels

def select_models(*labels):
    if len(labels) == 1 and labels[0] in MODEL_PRESETS:
        labels = MODEL_PRESETS[labels[0]]

    return [(label, MODEL_BANK[label]) for label in labels]

def series_labels(series):
    return [label for label, _ in series]

def legend_columns(series, maximum=4):
    return min(len(series), maximum)

MODEL_COLOR = {
    'Empirical': '#000000',
    MODEL_NAME: '#009E73',
    MODEL_T5_NAME: '#D55E00',
    HURDLE_AR5_NAME: '#0072B2'}

MODEL_LINESTYLE = {
    'Empirical': '-',
    MODEL_NAME: (0, (6, 2, 1, 2)),
    MODEL_T5_NAME: '--',
    HURDLE_AR5_NAME: '-.'}

def line_kwargs(label, **kwargs):
    return {'color': MODEL_COLOR.get(label),'linestyle': MODEL_LINESTYLE.get(label, '-'),**kwargs}


In [ ]:
raw = pd.read_csv(DATA/'df_traj_all.csv')
raw['CareerAgeZero'] = pd.to_numeric(raw['CareerAgeZero'], errors='coerce')
raw['pubs_adj'] = pd.to_numeric(raw['pubs_adj'], errors='coerce').clip(lower=0)
raw = raw.dropna(subset=['dblp_id', 'CareerAgeZero', 'pubs_adj'])

full_ids = (raw.groupby('dblp_id')['CareerAgeZero'].max().loc[lambda s: s.eq(Y)].index)

prepared = raw[raw['dblp_id'].isin(full_ids)].copy()
prepared = (prepared.groupby(['dblp_id', 'CareerAgeZero'], as_index=False)['pubs_adj'].sum())

panel = ( prepared.pivot(index='dblp_id', columns='CareerAgeZero', values='pubs_adj').reindex(columns=np.arange(Y + 1)).sort_index())

missing_person_years = int(panel.isna().sum().sum())
panel = panel.fillna(0.0)

Q_EMP = panel.to_numpy(dtype=float)
EMP_IDS = panel.index.to_numpy()
N_EMP = Q_EMP.shape[0]

register_model('Empirical', Q_EMP)


In [ ]:
STAGE_ORDER = ['0', '1-4', '5-7', '8-20']
STAGE_TRANSITIONS = {
    '0': np.array([0]),
    '1-4': np.arange(1, 5),
    '5-7': np.arange(5, 8),
    '8-20': np.arange(8, 20)}

def transition_stage(t):
    if t == 0:
        return '0'
    if 1 <= t <= 4:
        return '1-4'
    if 5 <= t <= 7:
        return '5-7'
    if 8 <= t <= 19:
        return '8-20'
    raise ValueError(f'No stage for transition year {t}')

def target_stage(target_year):
    return transition_stage(target_year - 1)

def history_panel(q, rho, max_lag=None):
    q = np.asarray(q, dtype=float)
    history = np.zeros_like(q, dtype=float)
    logged = np.log1p(q)

    for target_year in range(1, q.shape[1]):
        end = target_year - 1
        if end <= 0:
            continue
        start = 0 if max_lag is None else max(0, end - max_lag)
        block = logged[:, start:end]
        weights = rho ** np.arange(block.shape[1] - 1, -1, -1)
        history[:, target_year] = block @ weights / weights.sum()

    return history

def build_transition_rows(q, history):
    rows = []

    for target_year in range(1, q.shape[1]):
        transition_year = target_year - 1
        q_prev = q[:, transition_year]
        q_now = q[:, target_year]

        rows.append(pd.DataFrame({
            'scholar': np.arange(q.shape[0]),
            'transition_year': transition_year,'target_year': target_year,
            'stage': transition_stage(transition_year),
            'q_prev': q_prev,'q': q_now,
            'active': (q_now > 0).astype(int),
            'x_prev': np.log1p(q_prev),
            'prev_active': (q_prev > 0).astype(int),
            'restart': (q_prev <= 0).astype(int),
            'history': history[:, target_year]}))

    return pd.concat(rows, ignore_index=True)


In [ ]:
CONTINUOUS = {'x_prev', 'history'} | {f'x_lag{k}' for k in range(1, AR_ORDER + 1)}
RIDGE = 1e-6
MIN_PROB = 1e-9


def fit_scaler(data, feature_names):
    means = {}
    scales = {}

    for name in feature_names:
        if name in CONTINUOUS:
            means[name] = float(data[name].mean())
            scale = float(data[name].std(ddof=0))
            scales[name] = scale if np.isfinite(scale) and scale > 1e-8 else 1.0
        else:
            means[name] = 0.0
            scales[name] = 1.0
    return means, scales


def design_matrix(data, feature_names, means, scales):
    columns = [np.ones(len(data))]

    for name in feature_names:
        columns.append((data[name].to_numpy(dtype=float) - means[name]) / scales[name])

    return np.column_stack(columns)


def array_design(values, spec):
    n = len(next(iter(values.values())))
    columns = [np.ones(n)]

    for name in spec['feature_names']:
        columns.append((np.asarray(values[name], dtype=float) - spec['means'][name]) / spec['scales'][name])

    return np.column_stack(columns)


def fit_logistic(X, y, constrained_index=None):
    def objective(weights):
        eta = np.clip(X @ weights, -35, 35)
        p = expit(eta)
        nll = -np.sum(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))
        nll += RIDGE * np.sum(weights[1:] ** 2)

        gradient = X.T @ (p - y)
        gradient[1:] += 2 * RIDGE * weights[1:]
        return nll, gradient

    bounds = [(None, None)] * X.shape[1]
    if constrained_index is not None:
        bounds[constrained_index] = (0, None)

    result = minimize(objective, np.zeros(X.shape[1]), jac=True, method='L-BFGS-B', bounds=bounds)

    if not result.success:
        print(f'Logistic warning: {result.message}')

    return result.x


def fit_stage(stage_data, use_history=True, constrain_history=True):
    activity_features = ['x_prev', 'prev_active']
    positive_features = ['x_prev', 'restart']

    if use_history:
        activity_features.append('history')
        positive_features.append('history')

    act_means, act_scales = fit_scaler(stage_data, activity_features)
    X_act = design_matrix(stage_data, activity_features, act_means, act_scales)
    y_act = stage_data['active'].to_numpy(dtype=float)

    act_history_index = None
    if use_history and constrain_history:
        act_history_index = 1 + activity_features.index('history')

    activity_coef = fit_logistic(X_act, y_act, constrained_index=act_history_index)

    positive = stage_data[stage_data['active'].eq(1)].copy()
    pos_means, pos_scales = fit_scaler(positive, positive_features)
    X_pos = design_matrix(positive, positive_features, pos_means, pos_scales)
    y_pos = np.log(positive['q'].to_numpy(dtype=float))

    lower = np.full(X_pos.shape[1], -np.inf)
    upper = np.full(X_pos.shape[1], np.inf)

    if use_history and constrain_history:
        lower[1 + positive_features.index('history')] = 0.0

    positive_fit = lsq_linear(X_pos, y_pos, bounds=(lower, upper), lsq_solver='exact')

    positive_coef = positive_fit.x
    residuals = y_pos - X_pos @ positive_coef
    residuals = residuals - residuals.mean()
    sigma = max(float(np.sqrt(np.mean(residuals ** 2))), 1e-8)
    laplace_scale = max(float(np.mean(np.abs(residuals))), 1e-8)

    return {'activity': {
            'feature_names': activity_features,
            'means': act_means,
            'scales': act_scales,
            'coef': activity_coef,
            'n': len(stage_data)},
        'positive': {
            'feature_names': positive_features,
            'means': pos_means,
            'scales': pos_scales,
            'coef': positive_coef,
            'residuals': residuals,
            'sigma': sigma,
            'laplace_scale': laplace_scale,
            'n': len(positive)}}


def fit_model(q, history, use_history=True, constrain_history=True):
    rows = build_transition_rows(q, history)
    return {stage: fit_stage(rows[rows['stage'].eq(stage)], use_history=use_history, constrain_history=constrain_history) for stage in STAGE_ORDER}


def score_model(model, q, history):
    rows = build_transition_rows(q, history)
    nll = 0.0
    n = 0

    for stage in STAGE_ORDER:
        stage_data = rows[rows['stage'].eq(stage)]
        fitted = model[stage]

        X_act = design_matrix(stage_data, fitted['activity']['feature_names'], fitted['activity']['means'], fitted['activity']['scales'])
        p = np.clip(expit(X_act @ fitted['activity']['coef']), MIN_PROB, 1 - MIN_PROB)
        y = stage_data['active'].to_numpy(dtype=float)
        nll -= np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
        n += len(stage_data)

        positive = stage_data[stage_data['active'].eq(1)]
        X_pos = design_matrix(positive, fitted['positive']['feature_names'], fitted['positive']['means'], fitted['positive']['scales'])
        y_pos = np.log(positive['q'].to_numpy(dtype=float))
        sigma = fitted['positive']['sigma']
        standardized = (y_pos - X_pos @ fitted['positive']['coef']) / sigma

        nll += np.sum(0.5 * standardized ** 2 + np.log(sigma) + 0.5 * np.log(2 * np.pi))

    return nll / n


def original_scale_coefficients(spec):
    names = spec['feature_names']
    standardized = spec['coef']
    original = {'intercept': float(standardized[0])}

    for j, name in enumerate(names, start=1):
        original[name] = float(standardized[j] / spec['scales'][name])
        original['intercept'] -= standardized[j] * spec['means'][name] / spec['scales'][name]

    return original


def build_hurdle_ar_rows(q, order=AR_ORDER):
    rows = []

    for target_year in range(1, q.shape[1]):
        transition_year = target_year - 1
        q_prev = q[:, transition_year]
        q_now = q[:, target_year]
        data = {
            'scholar': np.arange(q.shape[0]),
            'transition_year': transition_year,
            'target_year': target_year,
            'stage': transition_stage(transition_year),
            'q_prev': q_prev,
            'q': q_now,
            'prev_active': (q_prev > 0).astype(int),
            'active': (q_now > 0).astype(int)}

        for lag in range(1, order + 1):
            lag_year = transition_year - lag + 1
            data[f'x_lag{lag}'] = np.log1p(q[:, lag_year]) if lag_year >= 0 else np.zeros(q.shape[0])

        rows.append(pd.DataFrame(data))

    return pd.concat(rows, ignore_index=True)


def ar_design(data, feature_names):
    return np.column_stack([np.ones(len(data))] + [data[name].to_numpy(dtype=float) for name in feature_names])


def ar_value_design(values, feature_names):
    return np.column_stack([np.ones(len(next(iter(values.values()))))] + [np.asarray(values[name], dtype=float) for name in feature_names])


def fit_hurdle_ar(q, order=AR_ORDER):
    rows = build_hurdle_ar_rows(q, order=order)
    ar_feature_names = [f'x_lag{k}' for k in range(1, order + 1)]
    activity_features = ['x_lag1', 'prev_active']
    fitted = {}

    for stage in STAGE_ORDER:
        d = rows[rows['stage'].eq(stage)].copy()

        act_means, act_scales = fit_scaler(d, activity_features)
        X_act = design_matrix(d, activity_features, act_means, act_scales)
        y_act = d['active'].to_numpy(dtype=float)
        activity_coef = fit_logistic(X_act, y_act)

        continuation = d[d['prev_active'].eq(1) & d['active'].eq(1)].copy()
        X = ar_design(continuation, ar_feature_names)
        y = np.log(continuation['q'].to_numpy(dtype=float))
        coef = np.linalg.lstsq(X, y, rcond=None)[0]
        residuals = y - X @ coef
        residuals = residuals - residuals.mean()
        sigma = max(float(np.sqrt(np.mean(residuals ** 2))), 1e-8)
        laplace_scale = max(float(np.mean(np.abs(residuals))), 1e-8)

        restarts = d[d['prev_active'].eq(0) & d['active'].eq(1)]['q'].to_numpy(dtype=float)
        restart_scale = float(restarts.mean()) if len(restarts) else float(q[q > 0].mean())

        fitted[stage] = {
            'activity': {
                'feature_names': activity_features,
                'means': act_means,
                'scales': act_scales,
                'coef': activity_coef,
                'n': len(d)},
            'positive': {
                'feature_names': ar_feature_names,
                'coef': coef,
                'residuals': residuals,
                'sigma': sigma,
                'laplace_scale': laplace_scale,
                'n': len(continuation)},
            'restart_scale': max(restart_scale, 1e-8),
            'restart_n': len(restarts)}

    q0 = q[:, 0]
    positive_q0 = q0[q0 > 0]
    fitted['initialization'] = {'p_active': float(np.mean(q0 > 0)), 'positive_scale': float(positive_q0.mean())}
    fitted['order'] = order

    return fitted


def score_hurdle_ar(model, q, order=AR_ORDER):
    rows = build_hurdle_ar_rows(q, order=order)
    nll = 0.0
    n = 0

    for stage in STAGE_ORDER:
        d = rows[rows['stage'].eq(stage)].copy()
        active = d['active'].to_numpy(dtype=float)

        act = model[stage]['activity']
        X_act = design_matrix(d, act['feature_names'], act['means'], act['scales'])
        p_active = np.clip(expit(X_act @ act['coef']), MIN_PROB, 1 - MIN_PROB)
        nll -= np.sum(active * np.log(p_active) + (1 - active) * np.log(1 - p_active))
        n += len(d)

        continuation = d[d['prev_active'].eq(1) & d['active'].eq(1)]
        if len(continuation):
            pos = model[stage]['positive']
            X = ar_design(continuation, pos['feature_names'])
            y = np.log(continuation['q'].to_numpy(dtype=float))
            resid = y - X @ pos['coef']
            sigma = pos['sigma']
            nll += np.sum(0.5 * (resid / sigma) ** 2 + np.log(sigma) + 0.5 * np.log(2 * np.pi))

        restart = d[d['prev_active'].eq(0) & d['active'].eq(1)]['q'].to_numpy(dtype=float)
        if len(restart):
            scale = model[stage]['restart_scale']
            nll += np.sum(np.log(scale) + restart / scale)

    return nll / n


In [ ]:
folds = list(KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(Q_EMP))

def mean_cv_hurdle_ar(order=AR_ORDER):
    scores = []

    for train_idx, test_idx in folds:
        fitted = fit_hurdle_ar(Q_EMP[train_idx], order=order)
        scores.append(score_hurdle_ar(fitted, Q_EMP[test_idx], order=order))

    return float(np.mean(scores))

def profile_self_exciting(max_lag=None):
    profile_rows = []

    for rho in RHO_GRID:
        history = history_panel(Q_EMP, rho, max_lag=max_lag)
        scores = []

        for train_idx, test_idx in folds:
            fold_model = fit_model(Q_EMP[train_idx], history[train_idx], use_history=True, constrain_history=CONSTRAIN_EXCITATION)
            scores.append(score_model(fold_model, Q_EMP[test_idx], history[test_idx]))

        profile_rows.append({'rho': rho, 'mean_nll': np.mean(scores), 'se_nll': np.std(scores, ddof=1) / np.sqrt(len(scores))})

    return pd.DataFrame(profile_rows)

def memory_half_life(rho):
    if 0 < rho < 1:
        return float(np.log(0.5) / np.log(rho))
    if rho == 0:
        return 0.0
    return np.inf

rho_profile_s = profile_self_exciting(max_lag=None).assign(model=MODEL_NAME, max_lag=np.nan)
rho_profile_t5 = profile_self_exciting(max_lag=TRUNCATED_HISTORY_LENGTH).assign(model=MODEL_T5_NAME, max_lag=TRUNCATED_HISTORY_LENGTH)
rho_profile = pd.concat([rho_profile_s, rho_profile_t5], ignore_index=True)

RHO_HAT = float(rho_profile_s.loc[rho_profile_s['mean_nll'].idxmin(), 'rho'])
RHO_T5_HAT = float(rho_profile_t5.loc[rho_profile_t5['mean_nll'].idxmin(), 'rho'])
HALF_LIFE = memory_half_life(RHO_HAT)
HALF_LIFE_T5 = memory_half_life(RHO_T5_HAT)

HURDLE_AR5_CV_NLL = mean_cv_hurdle_ar(order=AR_ORDER)
SELF_EXCITING_CV_NLL = float(rho_profile_s['mean_nll'].min())
SELF_EXCITING_T5_CV_NLL = float(rho_profile_t5['mean_nll'].min())

rho_profile.to_csv(RESULT_DIR / 'rho_profile.csv', index=False)


In [ ]:
H_EMP = history_panel(Q_EMP, RHO_HAT)
H_EMP_T5 = history_panel(Q_EMP, RHO_T5_HAT, max_lag=TRUNCATED_HISTORY_LENGTH)

hurdle_ar5_model = fit_hurdle_ar(Q_EMP, order=AR_ORDER)
self_exciting_model = fit_model(Q_EMP, H_EMP, use_history=True, constrain_history=True)
self_exciting_t5_model = fit_model(Q_EMP, H_EMP_T5, use_history=True, constrain_history=True)

def parameter_table(model, model_label):
    rows = []

    for stage in STAGE_ORDER:
        for equation in ['activity', 'positive']:
            spec = model[stage][equation]
            original = original_scale_coefficients(spec)
            row = {'model': model_label, 'stage': stage, 'equation': equation, 'n': spec['n'], **original}

            if equation == 'positive':
                row['sigma'] = spec['sigma']
                row['laplace_scale'] = spec['laplace_scale']
            rows.append(row)

    return pd.DataFrame(rows)

def hurdle_ar_parameter_table(model, model_label=HURDLE_AR5_NAME):
    rows = []

    for stage in STAGE_ORDER:
        act = model[stage]['activity']
        pos = model[stage]['positive']
        act_original = original_scale_coefficients(act)
        row = {
            'model': model_label,
            'stage': stage,
            'activity_n': act['n'],
            'positive_n': pos['n'],
            'restart_scale': model[stage]['restart_scale'],
            'restart_n': model[stage]['restart_n'],
            'positive_intercept': pos['coef'][0],
            'sigma': pos['sigma'],
            'laplace_scale': pos['laplace_scale']}

        row.update({f'activity_{name}': value for name, value in act_original.items()})
        row.update({name: coef for name, coef in zip(pos['feature_names'], pos['coef'][1:])})
        rows.append(row)

    return pd.DataFrame(rows)

hurdle_ar5_params = hurdle_ar_parameter_table(hurdle_ar5_model)
hurdle_ar5_params.to_csv(RESULT_DIR / 'hurdle_ar5_s_p_stage_parameters.csv', index=False)

# display(hurdle_ar5_params)
# params


In [ ]:
rng = np.random.default_rng(SEED)
bootstrap_rows = []

for b in range(N_BOOT):
    sampled = rng.integers(0, N_EMP, size=N_EMP)
    q_boot = Q_EMP[sampled]

    for label, rho, max_lag in [(MODEL_NAME, RHO_HAT, None), (MODEL_T5_NAME, RHO_T5_HAT, TRUNCATED_HISTORY_LENGTH)]:
        h_boot = history_panel(q_boot, rho, max_lag=max_lag)
        fitted = fit_model(q_boot, h_boot, use_history=True, constrain_history=True)
        table = parameter_table(fitted, label)
        table['bootstrap'] = b
        table['rho'] = rho
        table['max_lag'] = np.nan if max_lag is None else max_lag
        bootstrap_rows.append(table)

bootstrap_params = pd.concat(bootstrap_rows, ignore_index=True)
bootstrap_params.to_csv(RESULT_DIR / 'history_coefficient_bootstrap.csv', index=False)

bootstrap_ci = (bootstrap_params.groupby(['model', 'stage', 'equation'])['history'].quantile([0.025, 0.5, 0.975]).unstack().reset_index().rename(columns={0.025: 'low', 0.5: 'median', 0.975: 'high'}))
bootstrap_ci.to_csv(RESULT_DIR / 'history_coefficient_bootstrap_ci.csv', index=False)


In [ ]:
def draw_residuals(spec, size, rng, distribution):
    if distribution == 'empirical':
        return rng.choice(spec['residuals'], size=size, replace=True)
    if distribution == 'normal':
        return rng.normal(0, spec['sigma'], size=size)
    if distribution == 'laplace':
        return rng.laplace(0, spec['laplace_scale'], size=size)
    raise ValueError(f'Unknown residual distribution: {distribution}')

def positive_exponential(scale, size, rng):
    draws = rng.exponential(scale=scale, size=size)
    zero = draws <= 0
    while zero.any():
        draws[zero] = rng.exponential(scale=scale, size=zero.sum())
        zero = draws <= 0
    return draws

def history_for_target(simulated, target_year, rho, max_lag=None):
    end = target_year - 1
    if end <= 0:
        return np.zeros(simulated.shape[0])
    start = 0 if max_lag is None else max(0, end - max_lag)
    block = np.log1p(simulated[:, start:end])
    weights = rho ** np.arange(block.shape[1] - 1, -1, -1)
    return block @ weights / weights.sum()

def simulate_self_exciting(model, rho, n_sim, seed, max_lag=None):
    rng = np.random.default_rng(seed)
    simulated = np.zeros((n_sim, Y + 1), dtype=float)
    simulated[:, 0] = rng.choice(Q_EMP[:, 0], size=n_sim, replace=True)

    for target_year in range(1, Y + 1):
        history = history_for_target(simulated, target_year, rho, max_lag=max_lag)
        q_prev = simulated[:, target_year - 1]
        values = {'x_prev': np.log1p(q_prev), 'prev_active': (q_prev > 0).astype(float), 'restart': (q_prev <= 0).astype(float), 'history': history}

        stage = target_stage(target_year)
        activity = model[stage]['activity']
        X_act = array_design(values, activity)
        p_active = expit(np.clip(X_act @ activity['coef'], -35, 35))
        active = rng.random(n_sim) < p_active

        if active.any():
            positive = model[stage]['positive']
            active_values = {name: values[name][active] for name in values}
            X_pos = array_design(active_values, positive)
            mean_log_q = X_pos @ positive['coef']
            noise = draw_residuals(positive, active.sum(), rng, SELF_RESIDUAL_DIST)
            log_q = np.clip(mean_log_q + noise, -30, 30)
            simulated[active, target_year] = np.exp(log_q)

    return simulated

def ar_values_from_simulated(simulated, transition_year, rows, order=AR_ORDER):
    values = {}

    for lag in range(1, order + 1):
        lag_year = transition_year - lag + 1
        values[f'x_lag{lag}'] = np.log1p(simulated[rows, lag_year]) if lag_year >= 0 else np.zeros(len(rows))

    return values

def simulate_hurdle_ar(model, n_sim, seed, order=AR_ORDER):
    rng = np.random.default_rng(seed)
    simulated = np.zeros((n_sim, Y + 1), dtype=float)

    init = model['initialization']
    active = rng.random(n_sim) < init['p_active']
    simulated[active, 0] = positive_exponential(init['positive_scale'], active.sum(), rng)

    for transition_year in range(Y):
        stage = transition_stage(transition_year)
        q_prev = simulated[:, transition_year]
        prev_active = (q_prev > 0).astype(int)

        activity = model[stage]['activity']
        activity_values = {'x_lag1': np.log1p(q_prev), 'prev_active': prev_active.astype(float)}
        X_act = array_design(activity_values, activity)
        prob_active = expit(np.clip(X_act @ activity['coef'], -35, 35))
        next_active = rng.random(n_sim) < prob_active

        continuation = next_active & (prev_active == 1)
        if continuation.any():
            pos = model[stage]['positive']
            rows = np.flatnonzero(continuation)
            values = ar_values_from_simulated(simulated, transition_year, rows, order=order)
            X_pos = ar_value_design(values, pos['feature_names'])
            mean_log_q = X_pos @ pos['coef']
            noise = draw_residuals(pos, continuation.sum(), rng, HURDLE_RESIDUAL_DIST)
            simulated[continuation, transition_year + 1] = np.exp(np.clip(mean_log_q + noise, -30, 30))

        restart = next_active & (prev_active == 0)
        if restart.any():
            simulated[restart, transition_year + 1] = positive_exponential(model[stage]['restart_scale'], restart.sum(), rng)

    return simulated

Q_SELF = simulate_self_exciting(self_exciting_model, RHO_HAT, N_SIM, SEED + 1)
Q_SELF_T5 = simulate_self_exciting(self_exciting_t5_model, RHO_T5_HAT, N_SIM, SEED + 2, max_lag=TRUNCATED_HISTORY_LENGTH)
Q_AR5 = simulate_hurdle_ar(hurdle_ar5_model, N_SIM, SEED + 3, order=AR_ORDER)

np.save(RESULT_DIR / f'{MODEL_TAG}_{MODEL_NAME.lower().replace("-", "_")}_trajs.npy', Q_SELF)
np.save(RESULT_DIR / f'{MODEL_TAG}_{MODEL_T5_NAME.lower().replace("-", "_")}_trajs.npy', Q_SELF_T5)
np.save(RESULT_DIR / f'{MODEL_TAG}_hurdle_ar5_s_p_trajs.npy', Q_AR5)


In [ ]:
Q_SELF = register_model(MODEL_NAME, Q_SELF)
Q_SELF_T5 = register_model(MODEL_T5_NAME, Q_SELF_T5)
Q_AR5 = register_model(HURDLE_AR5_NAME, Q_AR5)

In [ ]:
register_preset('core', *COMPARISON_LABELS)
register_preset('conditional dropout', *COMPARISON_LABELS)
register_preset('moments', *COMPARISON_LABELS)
register_preset('all', *COMPARISON_LABELS)
register_preset('broad', *COMPARISON_LABELS)
register_preset('conditional', *COMPARISON_LABELS)
register_preset('manuscript', *COMPARISON_LABELS)
register_preset('restart', *COMPARISON_LABELS)


In [ ]:
def year_stats(q, label):
    raw = q
    logged = np.log(raw + EPS)

    return pd.DataFrame({
        'model': label,
        'year': np.arange(q.shape[1]),
        'mean': raw.mean(axis=0),
        'median': np.median(raw, axis=0),
        'variance': raw.var(axis=0),
        'zero_fraction': (raw == 0).mean(axis=0),
        'log_mean': logged.mean(axis=0),
        'log_median': np.median(logged, axis=0),
        'log_variance': logged.var(axis=0)})

moment_series = select_models('moments')
moment_stats = pd.concat([year_stats(q, label) for label, q in moment_series],ignore_index=True)

moment_stats.to_csv(RESULT_DIR / 'year_stats.csv', index=False)

In [ ]:
metrics = [
    ('mean', r'$\mathbf{A.}$ Raw mean'),
    ('median', r'$\mathbf{B.}$ Raw median'),
    ('variance', r'$\mathbf{C.}$ Raw variance'),
    ('log_mean', r'$\mathbf{D.}$ Log mean'),
    ('log_median', r'$\mathbf{E.}$ Log median'),
    ('log_variance', r'$\mathbf{F.}$ Log variance')]

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True)

for ax, (metric, title) in zip(axes.ravel(), metrics):
    for label, group in moment_stats.groupby('model', sort=False):
        ax.plot(group['year'],group[metric],label=label,**line_kwargs(label,linewidth=2))

    ax.set(title=title, xlabel='Career age')
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlim(left=0, right=20)
    ax.set_xticks(np.arange(0, 21, 1))
    ax.set_xticks(np.arange(0, 21, 0.5), minor=True)

    ax.grid(True, which='major', linewidth=0.8)
    ax.grid(True, which='minor', linewidth=0.4, alpha=0.5)
    ax.tick_params(axis='x', which='major', length=3)

axes[0, 0].set_ylabel('Productivity')
axes[1, 0].set_ylabel('Log productivity')

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(0.5, 1.02),ncol=legend_columns(moment_series)+1,frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / 'moment_trajectories.png', dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
moment_pairs = [
    ('mean', 'log_mean', r'$\mathbf{A.}$ Raw mean', r'$\mathbf{D.}$ Log mean', 'mean'),
    ('median', 'log_median', r'$\mathbf{B.}$ Raw median', r'$\mathbf{E.}$ Log median', 'median'),
    ('variance', 'log_variance', r'$\mathbf{C.}$ Raw variance', r'$\mathbf{F.}$ Log variance', 'variance')]

for raw_metric, log_metric, raw_title, log_title, name in moment_pairs:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)

    for ax, metric, title, ylabel in zip(axes,[raw_metric, log_metric],[raw_title, log_title],['Productivity', 'Log productivity']):
        for label, group in moment_stats.groupby('model', sort=False):
            ax.plot(group['year'], group[metric], label=label, **line_kwargs(label, linewidth=2))

        ax.set(title=title, xlabel='Career age', ylabel=ylabel)
        ax.spines[['top', 'right']].set_visible(False)
        ax.set_xlim(left=0, right=20)
        ax.set_xticks(np.arange(0, 21, 1))
        ax.set_xticks(np.arange(0, 21, 0.5), minor=True)

        ax.grid(True, which='major', linewidth=0.8)
        ax.grid(True, which='minor', linewidth=0.4, alpha=0.5)
        ax.tick_params(axis='x', which='major', length=3)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(0.5, 1.05),
        ncols=legend_columns(moment_series)+1,
        frameon=False)

    fig.tight_layout()
    fig.savefig(FIG_DIR / f'{name}_trajectories.png', dpi=DPI, bbox_inches='tight')
    plt.show()

In [ ]:
def percentile_ranks(q):
    return np.column_stack([
        pd.Series(q[:, t]).rank(method='average', pct=True).to_numpy()
        for t in range(q.shape[1])])

def rank_curve(q):
    ranks = percentile_ranks(q)

    return np.array([
        stats.spearmanr(ranks[:, 0], ranks[:, t]).statistic
        for t in range(q.shape[1])])

def rank_replicates(simulator, seed):
    persistence = np.empty((N_RANK_REPS, Y + 1))

    for rep in range(N_RANK_REPS):
        simulated = simulator(N_EMP, seed + rep)
        persistence[rep] = rank_curve(simulated)

    return persistence

rank_simulators = {
    MODEL_NAME: lambda n, seed: simulate_self_exciting(self_exciting_model, RHO_HAT, n, seed),
    MODEL_T5_NAME: lambda n, seed: simulate_self_exciting(self_exciting_t5_model, RHO_T5_HAT, n, seed, max_lag=TRUNCATED_HISTORY_LENGTH),
    HURDLE_AR5_NAME: lambda n, seed: simulate_hurdle_ar(hurdle_ar5_model, n, seed, order=AR_ORDER)}

emp_persistence = rank_curve(Q_EMP)
rank_draws = {label: rank_replicates(simulator, SEED + 1000 * (i + 1)) for i, (label, simulator) in enumerate(rank_simulators.items())}

years = np.arange(Y + 1)

fig, ax = plt.subplots(figsize=(7.2, 4.8))

ax.plot(years, emp_persistence, label='Empirical', **line_kwargs('Empirical', linewidth=2.6))

for label, draws in rank_draws.items():
    line, = ax.plot(years, draws.mean(axis=0), label=label, **line_kwargs(label, linewidth=2))
    ax.fill_between(years, np.quantile(draws, 0.025, axis=0), np.quantile(draws, 0.975, axis=0), alpha=0.15, color=line.get_color())

ax.axhline(0, linestyle='dotted', alpha=0.5, color='black')

ax.set(title='Rank mixing', xlabel='Career age', ylabel='Spearman correlation with year 0 rank', xlim=(0, 20), ylim=(-0.2, None))

ax.set_xticks(np.arange(0, 21, 1))
ax.set_xticks(np.arange(0, 21, 0.5), minor=True)

ax.grid(True, which='major', linewidth=0.8)
ax.grid(True, which='minor', linewidth=0.4, alpha=0.5)
ax.tick_params(axis='x', which='major', length=3)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=True)

fig.tight_layout()
fig.savefig(FIG_DIR / 'rank_mixing.png', dpi=DPI, bbox_inches='tight')
plt.show()

rank_rows = [{'model': 'Empirical', 'terminal_persistence': emp_persistence[-1], 'persistence_rmse': 0.0}]

for label, draws in rank_draws.items():
    mean_curve = draws.mean(axis=0)
    rank_rows.append({'model': label, 'terminal_persistence': mean_curve[-1], 'persistence_rmse': np.sqrt(np.mean((mean_curve - emp_persistence) ** 2))})

rank_summary = pd.DataFrame(rank_rows)
rank_summary.to_csv(RESULT_DIR / 'rank_summary.csv', index=False)


In [ ]:
INCREMENT_STAGES = {
    'transition 0': np.array([0]),
    'transitions 1-4': np.arange(1, 5),
    'transitions 5-7': np.arange(5, 8),
    'transitions 8-19': np.arange(8, Y)}

def laplace_fit(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    mu = np.median(values)
    scale = max(float(np.mean(np.abs(values - mu))), 1e-12)
    theoretical, observed = stats.probplot(values,dist=stats.laplace,sparams=(mu, scale),fit=False)
    qq_r = stats.pearsonr(theoretical, observed).statistic
    return mu, scale, qq_r, len(values)

def stage_increments(q, years=None):
    increments = np.diff(q, axis=1)
    if years is None:
        return increments.ravel()
    return increments[:, years].ravel()

def plot_laplace_hist(ax, values, model_label, subtitle, log_y=True):
    mu, scale, qq_r, n = laplace_fit(values)
    low, high = np.quantile(values, [0.002, 0.998])
    grid = np.linspace(low, high, 600)
    density = stats.laplace.pdf(grid, loc=mu, scale=scale)

    ax.hist(values,bins=80,density=True,alpha=0.65,range=(low, high),color=MODEL_COLOR.get(model_label))
    ax.plot(grid,density,linestyle='--',linewidth=2,color=MODEL_COLOR.get(model_label))
    ax.set_xlim(low, high)

    if log_y:
        ax.set_yscale('log')

    ax.set_title(f'{model_label}\n{subtitle}; ' + rf'$r_{{\mathrm{{QQ}}}}={qq_r:.3f}$')
    ax.spines[['top', 'right']].set_visible(False)

    return {'model': model_label,'stage': subtitle,'n': n,'mu': mu,'scale': scale,'qq_r': qq_r}

laplace_series = select_models('broad')
laplace_rows = []

fig, axes = plt.subplots(1,len(laplace_series),figsize=(3.5 * len(laplace_series), 4.4),squeeze=False)

for ax, (model_label, q) in zip(axes[0], laplace_series):
    laplace_rows.append(plot_laplace_hist(ax,stage_increments(q),model_label,'all transitions',log_y=True))
    ax.set_xlabel('Raw productivity increment')

axes[0, 0].set_ylabel('Log density')
fig.suptitle('Pooled raw-increment Laplace fits', y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / 'raw_increment_laplace_pooled.png',dpi=DPI,bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(len(INCREMENT_STAGES),len(laplace_series),figsize=(4.2 * len(laplace_series), 13),squeeze=False)

for row, (stage_label, stage_years) in enumerate(INCREMENT_STAGES.items()):
    for col, (model_label, q) in enumerate(laplace_series):
        ax = axes[row, col]
        laplace_rows.append(plot_laplace_hist(ax,stage_increments(q, stage_years),model_label,stage_label,log_y=True))
        if col == 0:
            ax.set_ylabel('Log density')
        if row == len(INCREMENT_STAGES) - 1:
            ax.set_xlabel('Raw productivity increment')

fig.suptitle('Stagewise raw-increment Laplace fits', y=1.005)
fig.tight_layout()
fig.savefig(FIG_DIR / 'raw_increment_laplace_stagewise.png',dpi=DPI,bbox_inches='tight')
plt.show()

laplace_summary = pd.DataFrame(laplace_rows)
laplace_summary.to_csv(RESULT_DIR / 'raw_increment_laplace.csv',index=False)

In [ ]:
def bowley_skewness(x):
    q25, q50, q75 = np.quantile(x, [0.25, 0.50, 0.75])
    denominator = q75 - q25
    return (q75 + q25 - 2 * q50) / denominator if denominator > 0 else np.nan

def tail_asymmetry(x):
    q05, q50, q95 = np.quantile(x, [0.05, 0.50, 0.95])
    lower = q50 - q05
    upper = q95 - q50
    denominator = upper + lower
    return (upper - lower) / denominator if denominator > 0 else np.nan

def bootstrap_ci(x, statistic, n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    estimates = np.array([statistic(rng.choice(x, size=len(x), replace=True)) for _ in range(n_boot)])
    return np.quantile(estimates, [0.025, 0.975])

def cumulative_lognormal(q):
    cumulative = q.sum(axis=1)
    logged = np.log2(cumulative[cumulative > 0])

    mu = logged.mean()
    sigma = logged.std(ddof=0)

    theoretical, observed = stats.probplot(logged, dist="norm",sparams=(mu, sigma),fit=False)

    qq_r = stats.pearsonr(theoretical, observed).statistic

    skew = stats.skew(logged, bias=False)
    skew_z, skew_p_left = stats.skewtest(logged,alternative="less",)
    bowley = bowley_skewness(logged)
    tail_asym = tail_asymmetry(logged)

    bowley_low, bowley_high = bootstrap_ci(logged,bowley_skewness,)

    tail_low, tail_high = bootstrap_ci(logged,tail_asymmetry)

    return {
        "logged": logged,
        "mu": mu,"sigma": sigma,
        "qq_r": qq_r,
        "skew": skew,"skew_z": skew_z,"skew_p_left": skew_p_left,
        "bowley_skew": bowley,"bowley_ci_low": bowley_low,"bowley_ci_high": bowley_high,
        "tail_asymmetry": tail_asym,"tail_ci_low": tail_low,"tail_ci_high": tail_high}

lognormal_series = select_models("broad")
lognormal_rows = []

fig, axes = plt.subplots(1,len(lognormal_series),figsize=(3 * len(lognormal_series), 3.5),squeeze=False,)

for ax, (model_label, q) in zip(axes[0], lognormal_series):
    result = cumulative_lognormal(q)

    logged = result["logged"]
    mu = result["mu"]
    sigma = result["sigma"]

    low, high = np.quantile(logged, [0.002, 0.998])
    grid = np.linspace(low, high, 500)

    ax.hist(logged,bins=50,density=True,alpha=0.75,color=MODEL_COLOR.get(model_label))

    ax.plot(grid,stats.norm.pdf(grid, mu, sigma),linestyle="--",linewidth=2,color=MODEL_COLOR.get(model_label))

    ax.set(title=(rf"{model_label}""\n"rf"$r_{{\mathrm{{QQ}}}}={result['qq_r']:.3f}$, "rf"$g_1={result['skew']:.2f}$"),xlabel=r"$\log_2$ cumulative productivity")

    ax.spines[["top", "right"]].set_visible(False)

    lognormal_rows.append({"model": model_label,
        "n": len(logged),"log_mu": mu,"log_sigma": sigma,
        "qq_r": result["qq_r"],
        "skew": result["skew"],"skew_z": result["skew_z"],"skew_p_left": result["skew_p_left"],
        "bowley_skew": result["bowley_skew"],"bowley_ci_low": result["bowley_ci_low"],"bowley_ci_high": result["bowley_ci_high"],
        "tail_asymmetry": result["tail_asymmetry"],"tail_ci_low": result["tail_ci_low"], "tail_ci_high": result["tail_ci_high"],})

axes[0, 0].set_ylabel("Density")

fig.tight_layout()
fig.savefig(FIG_DIR / "cumulative_lognormal.png",dpi=DPI,bbox_inches="tight")

plt.show()

lognormal_summary = pd.DataFrame(lognormal_rows)
lognormal_summary["skew_p_fdr"] = multipletests(lognormal_summary["skew_p_left"],method="fdr_bh",)[1]

lognormal_summary.to_csv(RESULT_DIR / "cumulative_lognormal.csv",index=False)


In [ ]:
def ecdf(values):
    x, counts = np.unique(np.asarray(values, dtype=float), return_counts=True)
    return x, np.cumsum(counts) / counts.sum()

def year_of_max(q, rng):
    q = q[q.max(axis=1) > 0]
    return np.array([rng.choice(np.flatnonzero(row == row.max())) for row in q])

def tmax_distribution(q):
    q = q[q.max(axis=1) > 0]
    p = np.zeros(q.shape[1])

    for row in q:
        maxima = np.flatnonzero(row == row.max())
        p[maxima] += 1 / len(maxima)

    return p / len(q)

def ks_against_empirical(empirical, simulated):
    result = stats.ks_2samp(empirical, simulated)
    return result.statistic, result.pvalue

cdf_series = select_models('moments')
max_rng = np.random.default_rng(SEED + 500)

cdf_values = {
    'year_max': {label: year_of_max(q, max_rng) for label, q in cdf_series},
    'cum5': {label: q[:, :6].sum(axis=1) for label, q in cdf_series},
    'zero_years': {label: (q == 0).sum(axis=1) for label, q in cdf_series}}

cdf_specs = [
    (r'$\mathbf{A.}$ Pr(reached year of $q_{\max}$)','Career age','year_max'),
    (r'$\mathbf{B.}$ Cumulative productivity by year 5','Cumulative productivity through year 5','cum5'),
    (r'$\mathbf{C.}$ CDF of zero years/career','Zero-productivity years per career','zero_years')]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.7))
ks_rows = []

for i, (ax, (title, xlabel, key)) in enumerate(zip(axes, cdf_specs)):
    empirical = cdf_values[key]['Empirical']

    for label, _ in cdf_series:
        values = cdf_values[key][label]
        x, y = ecdf(values)
        ax.step(x,y,where='post',label=label,**line_kwargs(label,linewidth=2))

        if label != 'Empirical':
            d, p = ks_against_empirical(empirical, values)
            ks_rows.append({'diagnostic': title,'model': label,'ks_D': d,'ks_p': p})

    ax.set(title=title,xlabel=xlabel,ylabel='Cumulative probability' if i == 0 else '',ylim=(0, 1.02))

    if i == 0:
        ax.set_xlim(0, Y)
        ax.xaxis.set_major_locator(MultipleLocator(1))
        ax.xaxis.set_minor_locator(MultipleLocator(0.5))
    elif i == 1:
        ax.set_xlim(left=0)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
        ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    else:
        ax.set_xlim(0, Y + 1)
        ax.xaxis.set_major_locator(MultipleLocator(1))
        ax.xaxis.set_minor_locator(MultipleLocator(0.5))

    ax.yaxis.set_major_locator(MultipleLocator(0.2))
    ax.yaxis.set_minor_locator(MultipleLocator(0.1))
    ax.grid(True, which='major', linewidth=0.8)
    ax.grid(True, which='minor', linewidth=0.4, alpha=0.5)
    ax.tick_params(axis='x', which='major', length=3)
    ax.tick_params(axis='y', which='major', length=3)
    ax.spines[['top', 'right']].set_visible(False)

    if i == 0:
        inset = inset_axes(ax,width='42%',height='39%',loc='lower right',borderpad=1.5)
        years = np.arange(Y + 1)

        for label, q in cdf_series:
            probability = tmax_distribution(q)
            inset.fill_between(years,probability,step='mid',alpha=0.12,color=MODEL_COLOR.get(label))
            inset.plot(years,probability,**line_kwargs(label,linewidth=1.5))

        inset.set_xlim(-0.2, Y + 0.2)
        inset.set_xlabel(r'$t$', fontsize=8, labelpad=0)
        inset.set_title(r'$\mathbf{D.}$ $P(t_{\max}=t)$', fontsize=9, pad=2)
        inset.xaxis.set_major_locator(MultipleLocator(5))
        inset.xaxis.set_minor_locator(MultipleLocator(1))
        inset.set_yticks([])
        inset.grid(True, which='major', linewidth=0.5, alpha=0.6)
        inset.grid(True, which='minor', linewidth=0.3, alpha=0.25)
        inset.tick_params(axis='x', which='major', labelsize=8, length=2)
        inset.tick_params(axis='x', which='minor', length=1)
        inset.xaxis.label.set_size(8)
        inset.xaxis.labelpad = 0
        inset.spines[['top', 'right']].set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(0.5, 1.06),ncol=legend_columns(cdf_series)+1,frameon=True)

fig.tight_layout()
fig.savefig(FIG_DIR / 'distributional_cdfs.png',dpi=DPI,bbox_inches='tight')
plt.show()

ks_summary = pd.DataFrame(ks_rows)
ks_summary.to_csv(RESULT_DIR / 'distributional_ks.csv',index=False)

In [ ]:
summary = pd.DataFrame({
    'model': [MODEL_NAME, MODEL_T5_NAME, HURDLE_AR5_NAME],
    'rho': [RHO_HAT, RHO_T5_HAT, np.nan],
    'max_lag': [np.nan, TRUNCATED_HISTORY_LENGTH, np.nan],
    'memory_half_life': [HALF_LIFE, HALF_LIFE_T5, np.nan],
    'cv_nll': [SELF_EXCITING_CV_NLL, SELF_EXCITING_T5_CV_NLL, HURDLE_AR5_CV_NLL],
    'terminal_rank_persistence': [
        rank_summary.loc[rank_summary['model'].eq(MODEL_NAME), 'terminal_persistence'].iloc[0],
        rank_summary.loc[rank_summary['model'].eq(MODEL_T5_NAME), 'terminal_persistence'].iloc[0],
        rank_summary.loc[rank_summary['model'].eq(HURDLE_AR5_NAME), 'terminal_persistence'].iloc[0]],
    'rank_persistence_rmse': [
        rank_summary.loc[rank_summary['model'].eq(MODEL_NAME), 'persistence_rmse'].iloc[0],
        rank_summary.loc[rank_summary['model'].eq(MODEL_T5_NAME), 'persistence_rmse'].iloc[0],
        rank_summary.loc[rank_summary['model'].eq(HURDLE_AR5_NAME), 'persistence_rmse'].iloc[0]]})

summary.to_csv(RESULT_DIR / 'model_summary.csv', index=False)

In [ ]:
def empirical_last_four():
    ordered = raw.copy()
    ordered['CareerAge'] = ordered['CareerAgeZero'].astype(int)
    ordered = ordered.sort_values(['dblp', 'CareerAge'])

    rolling4 = ordered.groupby('dblp')['pubs_adj'].rolling(4, min_periods=4).sum().reset_index(level=0, drop=True)

    rolling = (ordered.assign(RollingPubs=rolling4).dropna(subset=['RollingPubs']).groupby('dblp', as_index=False).agg(FourYearProd=('RollingPubs', 'last'), TotalYears=('CareerAge', 'max')))

    return rolling.loc[rolling['TotalYears'].gt(4) & rolling['FourYearProd'].gt(0), 'FourYearProd'].to_numpy(dtype=float)

def qq_details(productivity):
    productivity = np.asarray(productivity, dtype=float)
    productivity = productivity[np.isfinite(productivity) & (productivity > 0)]

    logged = np.log2(productivity)
    mu = logged.mean()
    sigma = logged.std(ddof=1)
    standardized = (logged - mu) / sigma

    (theoretical, ordered), (_, _, corr) = stats.probplot(standardized, dist='norm', fit=True, rvalue=True)

    return {'theoretical': theoretical, 'ordered': ordered, 'corr': float(corr), 'n': len(logged), 'log_mean': mu, 'log_sd': sigma}

emp_last_four = empirical_last_four()
last_four_by_model = {
    'Empirical': emp_last_four,
    MODEL_NAME: Q_SELF[:, 17:21].sum(axis=1),
    MODEL_T5_NAME: Q_SELF_T5[:, 17:21].sum(axis=1),
    HURDLE_AR5_NAME: Q_AR5[:, 17:21].sum(axis=1)}

qq_series = list(last_four_by_model.items())

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), gridspec_kw={'width_ratios': [2.2, 1]})

rows = []
line_colors = {}

for label, values in qq_series:
    details = qq_details(values)
    line, = axes[0].plot(details['theoretical'], details['ordered'], label=label, **line_kwargs(label, linewidth=2.2))
    line_colors[label] = line.get_color()

    rows.append({'model': label, 'window': 'Last four observed' if label == 'Empirical' else 'Years 17–20', 'n_positive': details['n'], 'qq_r': details['corr'], 'log2_mean': details['log_mean'], 'log2_sd': details['log_sd']})

axes[0].plot([-4, 4], [-4, 4], linewidth=1, linestyle='--', color='black', label='Normal reference')
axes[0].set(xlim=(-4, 4), ylim=(-4, 4), xlabel='Theoretical normal quantiles', ylabel='Standardized ordered log productivity', title=r'$\mathbf{A.}$ Four-year productivity QQ')

axes[0].xaxis.set_major_locator(MultipleLocator(1))
axes[0].xaxis.set_minor_locator(MultipleLocator(0.5))
axes[0].yaxis.set_major_locator(MultipleLocator(1))
axes[0].yaxis.set_minor_locator(MultipleLocator(0.5))

axes[0].grid(True, which='major', linewidth=0.8)
axes[0].grid(True, which='minor', linewidth=0.4, alpha=0.5)
axes[0].tick_params(which='major', length=3)
axes[0].legend(frameon=True)

qq_summary = pd.DataFrame(rows).sort_values('qq_r').reset_index(drop=True)

y = np.arange(len(qq_summary))
colors = [line_colors[label] for label in qq_summary['model']]

corr_min = qq_summary['qq_r'].min()
corr_max = qq_summary['qq_r'].max()
corr_pad = max((corr_max - corr_min) * 0.18, 0.002)

for correlation, color in zip(qq_summary['qq_r'], colors):
    axes[1].axvline(correlation, linewidth=1, alpha=0.25, color=color, zorder=1)

axes[1].scatter(qq_summary['qq_r'], y, s=55, c=colors, zorder=3)

axes[1].set_yticks(y, qq_summary['model'])
axes[1].yaxis.tick_right()
axes[1].set(xlim=(corr_min - corr_pad, min(1, corr_max + corr_pad)), ylim=(-0.6, len(y) - 0.4), xlabel='QQ correlation', title=r'$\mathbf{B.}$ Normality')

axes[1].xaxis.set_major_locator(MaxNLocator(nbins=5))
axes[1].xaxis.set_minor_locator(AutoMinorLocator(2))
axes[1].xaxis.set_major_formatter(FormatStrFormatter('%.3f'))

axes[1].grid(True, axis='x', which='major', linewidth=0.8)
axes[1].grid(True, axis='x', which='minor', linewidth=0.4, alpha=0.5)
axes[1].tick_params(axis='x', which='major', length=3)
axes[1].tick_params(axis='y', length=0)

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
fig.savefig(FIG_DIR / 'qq_last_four.png', dpi=DPI, bbox_inches='tight')
plt.show()

qq_summary.to_csv(RESULT_DIR / 'qq_last_four.csv', index=False)

In [ ]:
register_preset('conditional', *COMPARISON_LABELS)

In [ ]:
def conditional_dropout_rows(q, model_label):
    rows = []

    for t in range(Y):
        current = q[:, t]
        active = current > 0

        rows.append(pd.DataFrame({
            'model': model_label,
            'stage': transition_stage(t),
            'q_prev': current[active],
            'dropout': (q[active, t + 1] == 0).astype(int)}))

    return pd.concat(rows, ignore_index=True)

dropout_series = select_models('conditional')
dropout_labels = series_labels(dropout_series)
dropout_data = pd.concat([conditional_dropout_rows(q, label) for label, q in dropout_series],ignore_index=True)

dropout_rows = []

for stage in STAGE_ORDER:
    empirical = dropout_data[dropout_data['model'].eq('Empirical') & dropout_data['stage'].eq(stage)].copy()
    edges = np.unique(np.quantile(np.log1p(empirical['q_prev']),np.linspace(0, 1, 11)))
    empirical['q_bin'] = np.digitize(np.log1p(empirical['q_prev']),edges[1:-1],right=True)
    centers = empirical.groupby('q_bin')['q_prev'].mean()

    for model_label in dropout_labels:
        d = dropout_data[dropout_data['model'].eq(model_label) & dropout_data['stage'].eq(stage)].copy()
        d['q_bin'] = np.digitize(np.log1p(d['q_prev']),edges[1:-1],right=True)

        fitted = (d.groupby('q_bin').agg(dropout_count=('dropout', 'sum'),dropout_probability=('dropout', 'mean'),n=('dropout', 'size')).reset_index())
        fitted['q_mean'] = fitted['q_bin'].map(centers)

        p = fitted['dropout_probability']
        n = fitted['n']
        z = 1.96
        denominator = 1 + z**2 / n
        center = (p + z**2 / (2 * n)) / denominator
        half_width = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denominator

        fitted['low'] = (center - half_width).clip(0, 1)
        fitted['high'] = (center + half_width).clip(0, 1)
        fitted['model'] = model_label
        fitted['stage'] = stage
        dropout_rows.append(fitted)

conditional_dropout = pd.concat(dropout_rows, ignore_index=True).sort_values(['stage', 'model', 'q_mean'])

fig, axes = plt.subplots(1,4,figsize=(15, 3.5),sharey=False)

for ax, stage in zip(axes, STAGE_ORDER):
    stage_data = conditional_dropout[conditional_dropout['stage'].eq(stage)]

    for model_label in dropout_labels:
        d = stage_data[stage_data['model'].eq(model_label)].dropna(subset=['q_mean']).sort_values('q_mean')
        line, = ax.plot(d['q_mean'],d['dropout_probability'],label=model_label,**line_kwargs(model_label,marker='.',linewidth=2))

        if model_label == 'Empirical':
            ax.fill_between(d['q_mean'],d['low'],d['high'],alpha=0.15,color=line.get_color())

    ax.set(title=f'Stage {stage}',xlabel=r'Current productivity $q_t$')
    ax.set_ylim(bottom = 0)
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_ylabel(r'$P(q_{t+1}=0\mid q_t>0)$')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(0.5, 1.08),ncol=legend_columns(dropout_series)+2,frameon=False)
#fig.suptitle('Conditional dropout by productivity and career stage',y=1.18)
fig.tight_layout()
fig.savefig(FIG_DIR / 'conditional_dropout.png',dpi=DPI,bbox_inches='tight')
plt.show()

conditional_dropout.to_csv(RESULT_DIR / 'conditional_dropout.csv',index=False)


In [ ]:
def positive_career_sd(q):
    return np.array([row[row > 0].std(ddof=1) for row in q if (row > 0).sum() > 1])

register_preset('manuscript', *COMPARISON_LABELS)
marginal_series = select_models('manuscript')

emp_sd = Q_EMP.std(axis=1, ddof=1)
emp_sd_pos = positive_career_sd(Q_EMP)
emp_annual = Q_EMP[Q_EMP > 0]

q0_high = np.quantile(np.concatenate([q[:, 0] for _, q in marginal_series]), 0.995)
sd_high = np.quantile(np.concatenate([q.std(axis=1, ddof=1) for _, q in marginal_series]), 0.995)
sd_pos_high = np.quantile(np.concatenate([positive_career_sd(q) for _, q in marginal_series]), 0.995)
annual_high = np.quantile(np.concatenate([q[q > 0] for _, q in marginal_series]), 0.995)

marginal_rows = []
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))

for label, q in marginal_series:
    color = MODEL_COLOR.get(label)
    linestyle = MODEL_LINESTYLE.get(label, '-')
    q0 = q[:, 0]
    q0_scale = stats.expon.fit(q0, floc=0)[1]
    q0_grid = np.linspace(0, q0_high, 400)
    sd = q.std(axis=1, ddof=1)
    sd_pos = positive_career_sd(q)
    annual = q[q > 0]

    axes[0].hist(q0,bins=35,range=(0, q0_high),density=True,histtype='step',linewidth=2,color=color,linestyle=linestyle,label=label)
    axes[0].plot(q0_grid,stats.expon.pdf(q0_grid, scale=q0_scale),linestyle='--',linewidth=1.7,color=color)
    axes[1].hist(sd,bins=35,range=(0, sd_high),density=True,histtype='step',linewidth=2,color=color,linestyle=linestyle,label=label)
    axes[2].hist(sd_pos,bins=35,range=(0, sd_pos_high),density=True,histtype='step',linewidth=2,color=color,linestyle=linestyle,label=label)
    axes[3].hist(annual,bins=45,range=(0, annual_high),density=True,histtype='step',linewidth=2,color=color,linestyle=linestyle,label=f'{label} (zero={np.mean(q == 0):.1%})')

    sd_ks = stats.ks_2samp(emp_sd, sd)
    sd_pos_ks = stats.ks_2samp(emp_sd_pos, sd_pos)
    annual_ks = stats.ks_2samp(emp_annual, annual)

    marginal_rows.append({
        'model': label,
        'q0_exponential_scale': q0_scale,
        'q0_zero_fraction': np.mean(q0 == 0),
        'career_sd_mean': sd.mean(),
        'career_sd_median': np.median(sd),
        'career_sd_ks_D': sd_ks.statistic,
        'career_sd_ks_p': sd_ks.pvalue,
        'positive_career_sd_mean': sd_pos.mean(),
        'positive_career_sd_median': np.median(sd_pos),
        'positive_career_sd_ks_D': sd_pos_ks.statistic,
        'positive_career_sd_ks_p': sd_pos_ks.pvalue,
        'annual_zero_fraction': np.mean(q == 0),
        'annual_positive_ks_D': annual_ks.statistic,
        'annual_positive_ks_p': annual_ks.pvalue})

axes[0].set(title=r'$\mathbf{A.}$ First-year productivity',xlabel=r'$q_0$',ylabel='Log density')
axes[0].set_yscale('log')
axes[1].set(title=r'$\mathbf{B.}$ Within-career SD',xlabel='Career SD',ylabel='Density')
axes[2].set(title=r'$\mathbf{C.}$ Within-career SD, zeros omitted',xlabel='Career SD')
axes[3].set(title=r'$\mathbf{D.}$ Annual productivity, positive years',xlabel='Productivity',ylabel='Log density')
axes[3].set_yscale('log')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig(FIG_DIR / 'manuscript_marginal_checks.png', dpi=DPI, bbox_inches='tight')
plt.show()

marginal_summary = pd.DataFrame(marginal_rows)
marginal_summary.to_csv(RESULT_DIR / 'manuscript_marginal_checks.csv', index=False)

In [ ]:
def canonical_trajectories(q):
    n_years = q.shape[1]
    years = np.arange(n_years)

    X = np.column_stack([np.ones(n_years), years])
    coef = np.linalg.lstsq(X, q.T, rcond=None)[0]
    rss = np.sum((q.T - X @ coef) ** 2, axis=0)

    k = 3
    best_aicc = (n_years * np.log(np.maximum(rss, 1e-12) / n_years)+ 2 * k + 2 * k * (k + 1) / (n_years - k - 1))

    best_cp = np.full(q.shape[0], -1)
    first_slope = coef[1].copy()
    second_slope = coef[1].copy()

    for cp in range(3, 18):
        X = np.column_stack([np.ones(n_years),years,np.maximum(0, years - cp)])
        coef = np.linalg.lstsq(X, q.T, rcond=None)[0]
        rss = np.sum((q.T - X @ coef) ** 2, axis=0)

        k = 4
        aicc = (n_years * np.log(np.maximum(rss, 1e-12) / n_years)+ 2 * k + 2 * k * (k + 1) / (n_years - k - 1))

        better = aicc < best_aicc
        best_aicc[better] = aicc[better]
        best_cp[better] = cp
        first_slope[better] = coef[1, better]
        second_slope[better] = coef[1, better] + coef[2, better]

    canonical = ((best_cp >= 0) & (first_slope > 0) & (second_slope < 0) & (np.abs(first_slope) >= 2 * np.abs(second_slope)))

    return canonical, best_cp, first_slope, second_slope

canonical_series = select_models('broad')
canonical_rows = []

for label, q in canonical_series:
    canonical, cp, first, second = canonical_trajectories(q)
    interval = stats.binomtest(canonical.sum(),len(canonical)).proportion_ci(method='wilson')

    canonical_rows.append({
        'model': label,
        'n': len(canonical),
        'canonical_n': canonical.sum(),
        'canonical_fraction': canonical.mean(),
        'ci_low': interval.low,
        'ci_high': interval.high,
        'median_changepoint': np.median(cp[canonical])})

canonical_summary = pd.DataFrame(canonical_rows)

fig, ax = plt.subplots(figsize=(7, 4.6))
x = np.arange(len(canonical_summary))
y = canonical_summary['canonical_fraction']
yerr = np.vstack([y - canonical_summary['ci_low'],canonical_summary['ci_high'] - y])

ax.bar(x,y,yerr=yerr,capsize=4,color=[MODEL_COLOR.get(label) for label in canonical_summary['model']])
ax.set(xticks=x,xticklabels=canonical_summary['model'],ylabel='Fraction canonical',title='Individually canonical trajectories')
ax.tick_params(axis='x', rotation=12)
ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
fig.savefig(FIG_DIR / 'canonical_trajectory_fraction.png', dpi=DPI, bbox_inches='tight')
plt.show()

canonical_summary.to_csv(RESULT_DIR / 'canonical_trajectory_fraction.csv',index=False)

In [ ]:
increment_series = select_models('moments')

emp_prev = Q_EMP[:, :-1].ravel()
emp_prev = emp_prev[emp_prev > 0]
increment_edges = np.unique(np.quantile(np.log1p(emp_prev), np.linspace(0, 1, 11)))

increment_rows = []

for label, q in increment_series:
    q_prev = q[:, :-1].ravel()
    delta = np.diff(q, axis=1).ravel()
    keep = q_prev > 0
    q_prev = q_prev[keep]
    delta = delta[keep]

    q_bin = np.digitize(np.log1p(q_prev),increment_edges[1:-1],right=True) + 1

    for b in range(1, len(increment_edges)):
        values = delta[q_bin == b]
        mu = np.median(values)
        beta, loc, scale = stats.gennorm.fit(values)

        increment_rows.append({
            'model': label,
            'q_bin': b,
            'n': len(values),
            'q_prev_mean': q_prev[q_bin == b].mean(),
            'delta_median': mu,
            'laplace_scale': np.mean(np.abs(values - mu)),
            'gennorm_beta': beta,
            'gennorm_loc': loc,
            'gennorm_scale': scale})

increment_by_q = pd.DataFrame(increment_rows)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.3))

for label, _ in increment_series:
    d = increment_by_q[increment_by_q['model'].eq(label)]

    axes[0].plot(d['q_prev_mean'],d['delta_median'],label=label,**line_kwargs(label,marker='o',linewidth=2))
    axes[1].plot(d['q_prev_mean'],d['laplace_scale'],label=label,**line_kwargs(label,marker='o',linewidth=2))
    axes[2].plot(d['q_prev_mean'],d['gennorm_beta'],label=label,**line_kwargs(label,marker='o',linewidth=2))

axes[0].axhline(0, linestyle=':', color='black')
axes[2].axhline(1, linestyle=':', color='black', label='Laplace')

axes[0].set(title='A. Conditional increment median', ylabel=r'Median $\Delta q_t$')
axes[1].set(title='B. Conditional Laplace scale', ylabel='Mean absolute deviation')
axes[2].set(title='C. Conditional exponential-power shape', ylabel=r'$\hat{\beta}$')

for ax in axes:
    ax.set_xlabel(r'Mean current productivity $q_t$')
    ax.spines[['top', 'right']].set_visible(False)

axes[0].legend(frameon=False)
axes[2].legend(frameon=False)

fig.tight_layout()
fig.savefig(FIG_DIR / 'increment_shape_by_productivity.png', dpi=DPI, bbox_inches='tight')
plt.show()

increment_by_q.to_csv(RESULT_DIR / 'increment_shape_by_productivity.csv',index=False)

In [ ]:
register_preset('restart', *COMPARISON_LABELS)

In [ ]:
def restart_spell_rows(q, label):
    rows = []
    run = np.zeros(q.shape[0], dtype=int)

    for t in range(Y):
        zero = q[:, t] == 0
        run = np.where(zero, run + 1, 0)
        idx = np.flatnonzero(zero)

        rows.append(pd.DataFrame({'model': label,'stage': transition_stage(t),'zero_run': np.minimum(run[idx], 6),'restart': (q[idx, t + 1] > 0).astype(int)}))

    return pd.concat(rows, ignore_index=True)


restart_series = select_models('restart')

restart_data = pd.concat([restart_spell_rows(q, label) for label, q in restart_series],ignore_index=True)

restart_summary = (restart_data.groupby(['model', 'stage', 'zero_run']).agg(restart_probability=('restart', 'mean'),n=('restart', 'size')).reset_index())

p = restart_summary['restart_probability']
restart_summary['se'] = np.sqrt(p * (1 - p) / restart_summary['n'])
restart_summary['low'] = (p - 1.96 * restart_summary['se']).clip(0, 1)
restart_summary['high'] = (p + 1.96 * restart_summary['se']).clip(0, 1)

MIN_MARKER_AREA = 10
MAX_MARKER_AREA = 180
POINT_ALPHA = 0.38

n_min = restart_summary['n'].min()
n_max = restart_summary['n'].max()

def marker_area(n):
    n = np.asarray(n, dtype=float)
    transformed = np.log1p(n)
    low = np.log1p(n_min)
    high = np.log1p(n_max)

    if high == low:
        return np.full_like(transformed,(MIN_MARKER_AREA + MAX_MARKER_AREA) / 2)

    scaled = (transformed - low) / (high - low)
    return MIN_MARKER_AREA + scaled * (MAX_MARKER_AREA - MIN_MARKER_AREA)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.2), sharey=True)

model_handles = []

for ax, stage in zip(axes, STAGE_ORDER[1:]):
    stage_data = restart_summary[restart_summary['stage'].eq(stage)]

    for label, _ in restart_series:
        d = (stage_data[stage_data['model'].eq(label)].sort_values('zero_run'))
        if d.empty:
            continue
        kwargs = line_kwargs(label, linewidth=2)
        kwargs.pop('marker', None)

        line, = ax.plot(d['zero_run'], d['restart_probability'], label=label, **kwargs)
        sizes = marker_area(d['n'])
        ax.scatter(d['zero_run'],d['restart_probability'],s=sizes,marker='o',facecolors='white',edgecolors='none',zorder=3)
        ax.scatter(d['zero_run'],d['restart_probability'],s=sizes,marker='o',facecolors=to_rgba(line.get_color(), POINT_ALPHA),edgecolors=to_rgba(line.get_color(), 0.85),linewidth=0.8,zorder=4)

        # if ax is axes[0]:
        #     model_handles.append(
        #         Line2D([0],[0],label=label,color=line.get_color(),linestyle=line.get_linestyle(),linewidth=2,marker='o',markersize=6,markerfacecolor=to_rgba(line.get_color(),POINT_ALPHA),markeredgecolor=line.get_color(),markeredgewidth=0.8))
        if ax is axes[0]:
            model_handles.append(line)
        if label == 'Empirical':
            ax.fill_between(d['zero_run'],d['low'],d['high'],color=line.get_color(),alpha=0.15,linewidth=0,zorder=0)

    ax.set(title=f'Stage {stage}',xlabel='Current zero-run length',ylim=(0, 1))

    if stage == STAGE_ORDER[1]:
        ax.set_xticks(
            range(1, 6),['1', '2', '3', '4', '5'])
        ax.set_xlim(0.8, 5.2)
    else:
        ax.set_xticks(range(1, 7),['1', '2', '3', '4', '5', '6+'])
        ax.set_xlim(0.8, 6.2)

    ax.spines[['top', 'right']].set_visible(False)


axes[0].set_ylabel(r'$P(q_{t+1}>0\mid q_t=0)$')

fig.legend(handles=model_handles,loc='upper center',handlelength = 4.2,bbox_to_anchor=(0.5, 1.08),ncol=legend_columns(restart_series) + 1,frameon=False)

min_power = int(np.ceil(np.log10(max(1, n_min))))
max_power = int(np.floor(np.log10(n_max)))
size_values = 10 ** np.arange(min_power, max_power + 1)

if len(size_values) == 0:
    size_values = np.unique(np.round(np.geomspace(max(1, n_min), n_max, 3)).astype(int))

size_handles = [axes[-1].scatter([],[],s=marker_area([n])[0],marker='o',facecolors=to_rgba('0.35', POINT_ALPHA),edgecolors=to_rgba('0.35', 0.85),linewidth=0.8,label=fr'$n={n:,}$') for n in size_values]

axes[-1].legend(
    handles=size_handles,
    title='Observations per point',
    loc='upper right',
    frameon=False,
    fontsize=7.5,
    title_fontsize=8,
    markerscale=0.7,
    handletextpad=0.4,
    labelspacing=0.35,
    borderpad=0.2)

#fig.suptitle('Conditional restart by inactivity duration',y=1.17)

fig.tight_layout()

fig.savefig(FIG_DIR / 'conditional_restart_by_zero_run.png',dpi=DPI,bbox_inches='tight')

plt.show()

restart_summary.to_csv(RESULT_DIR / 'conditional_restart_by_zero_run.csv',index=False)

In [ ]:
QQ_YEAR = 15
annual_qq_series = select_models('broad')
annual_qq_rows = []

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for label, q in annual_qq_series:
    details = qq_details(q[:, QQ_YEAR])
    axes[0].plot(details['theoretical'],details['ordered'],label=label,**line_kwargs(label,linewidth=2))

    total = q.sum(axis=1)
    share = q.max(axis=1)[total > 0] / total[total > 0]
    x, y = ecdf(share)
    axes[1].step(x,y,where='post',label=label,**line_kwargs(label,linewidth=2))

    annual_qq_rows.append({
        'model': label,
        'year': QQ_YEAR,
        'annual_qq_r': details['corr'],
        'annual_positive_n': details['n'],
        'max_share_mean': share.mean(),
        'max_share_median': np.median(share),
        'max_share_90': np.quantile(share, 0.90)})

axes[0].plot([-4, 4], [-4, 4], linestyle='--', linewidth=1, color='black')
axes[0].set(xlim=(-4, 4),ylim=(-4, 4),xlabel='Theoretical normal quantiles',ylabel='Std. ordered log productivity',title=f'A. One-year productivity QQ, year {QQ_YEAR}')
axes[1].set(xlabel='Maximum-year share of career productivity',ylabel='Cumulative probability',title='B. Concentration in the maximum year')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(frameon=False)

fig.tight_layout()
fig.savefig(FIG_DIR / 'annual_qq_and_max_share.png', dpi=DPI, bbox_inches='tight')
plt.show()

annual_qq_summary = pd.DataFrame(annual_qq_rows)
annual_qq_summary.to_csv(RESULT_DIR / 'annual_qq_and_max_share.csv',index=False)

In [ ]:
def delta_moment_stats(q, label):
    raw_delta = np.diff(q, axis=1)
    log_delta = np.diff(np.log(q + EPS), axis=1)

    return pd.DataFrame({
        'model': label,
        'transition_year': np.arange(Y),
        'target_year': np.arange(1, Y + 1),
        'raw_delta_mean': raw_delta.mean(axis=0),
        'raw_delta_variance': raw_delta.var(axis=0),
        'log_delta_mean': log_delta.mean(axis=0),
        'log_delta_variance': log_delta.var(axis=0)})

delta_series = select_models('moments')
delta_stats = pd.concat([delta_moment_stats(q, label) for label, q in delta_series],ignore_index=True)

metrics = [
    ('raw_delta_mean', r'$\mathbf{A.}$ Mean raw increment', r'Mean $\Delta q_t$'),
    ('raw_delta_variance', r'$\mathbf{B.}$ Raw-increment variance', r'Variance of $\Delta q_t$'),
    ('log_delta_mean', r'$\mathbf{C.}$ Mean log increment', r'Mean $\Delta\log(q_t+\varepsilon)$'),
    ('log_delta_variance', r'$\mathbf{D.}$ Log-increment variance', r'Variance of $\Delta\log(q_t+\varepsilon)$')]

fig, axes = plt.subplots(2, 2, figsize=(11.5, 8), sharex=True)

for ax, (metric, title, ylabel) in zip(axes.ravel(), metrics):
    for label, group in delta_stats.groupby('model', sort=False):
        ax.plot(group['target_year'],group[metric],label=label,**line_kwargs(label,linewidth=2,marker='o',markersize=3.5))

    if metric.endswith('_mean'):
        ax.axhline(0, color='black', linestyle=':', linewidth=1, alpha=0.7)

    ax.set(title=title,ylabel=ylabel,xlim=(1, Y))
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.xaxis.set_minor_locator(MultipleLocator(0.5))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.grid(True, which='major', linewidth=0.8)
    ax.grid(True, which='minor', linewidth=0.4, alpha=0.5)
    ax.tick_params(which='major', length=3)
    ax.spines[['top', 'right']].set_visible(False)

for ax in axes[1]:
    ax.set_xlabel('Target career year')

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles,labels,loc='upper center',bbox_to_anchor=(0.5, 1.01),ncol=legend_columns(delta_series),frameon=True)
fig.suptitle('Trajectories of raw and log productivity increments',y=1.045)
fig.tight_layout()
fig.savefig(FIG_DIR / 'increment_moment_trajectories.png',dpi=DPI,bbox_inches='tight')
plt.show()

delta_stats.to_csv(RESULT_DIR / 'increment_moment_trajectories.csv',index=False)


In [ ]:
rmse_metrics = ['raw_delta_mean','raw_delta_variance','log_delta_mean','log_delta_variance']

empirical_stats = delta_stats.loc[delta_stats['model'].eq('Empirical'),['target_year'] + rmse_metrics].set_index('target_year')
rmse_rows = []

for model in series_labels(delta_series):
    if model == 'Empirical':
        continue

    model_stats = delta_stats.loc[delta_stats['model'].eq(model),['target_year'] + rmse_metrics].set_index('target_year').reindex(empirical_stats.index)
    rmse_rows.append({'model': model,**{metric: np.sqrt(np.mean((model_stats[metric] - empirical_stats[metric]) ** 2)) for metric in rmse_metrics}})

rmse_summary = pd.DataFrame(rmse_rows).rename(columns={
    'raw_delta_mean': 'RMSE: raw delta mean',
    'raw_delta_variance': 'RMSE: raw delta variance',
    'log_delta_mean': 'RMSE: log delta mean',
    'log_delta_variance': 'RMSE: log delta variance'})

print('\nIncrement-moment trajectory RMSE against empirical\n')
print(rmse_summary.to_string(index=False,float_format=lambda x: f'{x:.6f}'))

rmse_summary.to_csv(RESULT_DIR / 'increment_moment_rmse.csv',index=False)


In [ ]:
def bootstrap_ci(values, n_boot=N_GAP_BOOT, seed=SEED, alpha=0.05):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    rng = np.random.default_rng(seed)
    draws = rng.choice(values, size=(n_boot, len(values)), replace=True).mean(axis=1)
    return np.quantile(draws, [alpha / 2, 0.5, 1 - alpha / 2])

def repeated_cv_scores(n_repeats=N_REPEATED_CV):
    rows = []
    H_full = history_panel(Q_EMP, RHO_HAT)
    H_t5 = history_panel(Q_EMP, RHO_T5_HAT, max_lag=TRUNCATED_HISTORY_LENGTH)

    for repeat in range(n_repeats):
        kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED + repeat)

        for fold, (train_idx, test_idx) in enumerate(kfold.split(Q_EMP)):
            model_full = fit_model(Q_EMP[train_idx], H_full[train_idx], use_history=True, constrain_history=CONSTRAIN_EXCITATION)
            model_t5 = fit_model(Q_EMP[train_idx], H_t5[train_idx], use_history=True, constrain_history=CONSTRAIN_EXCITATION)
            model_ar5p = fit_hurdle_ar(Q_EMP[train_idx], order=AR_ORDER)

            rows.extend([
                {'repeat': repeat, 'fold': fold, 'model': MODEL_NAME, 'cv_nll': score_model(model_full, Q_EMP[test_idx], H_full[test_idx])},
                {'repeat': repeat, 'fold': fold, 'model': MODEL_T5_NAME, 'cv_nll': score_model(model_t5, Q_EMP[test_idx], H_t5[test_idx])},
                {'repeat': repeat, 'fold': fold, 'model': HURDLE_AR5_NAME, 'cv_nll': score_hurdle_ar(model_ar5p, Q_EMP[test_idx], order=AR_ORDER)}])

    return pd.DataFrame(rows)

def paired_gap_summary(cv_scores, comparisons):
    pivot = cv_scores.pivot_table(index=['repeat', 'fold'], columns='model', values='cv_nll')
    rows = []

    for left, right, hypothesis in comparisons:
        gap = pivot[left] - pivot[right]
        lo, med, hi = bootstrap_ci(gap, seed=SEED + len(rows))
        ttest = stats.ttest_1samp(gap, 0.0)

        try:
            wilcoxon_p = stats.wilcoxon(gap).pvalue
        except ValueError:
            wilcoxon_p = np.nan

        rows.append({
            'hypothesis': hypothesis,
            'gap': f'{left} minus {right}',
            'mean_gap': gap.mean(),
            'median_gap': gap.median(),
            'ci95_low': lo,
            'ci95_median': med,
            'ci95_high': hi,
            'paired_t_p': ttest.pvalue,
            'wilcoxon_p': wilcoxon_p,
            'practically_equivalent_at_tol': (lo > -PRACTICAL_NLL_TOL) and (hi < PRACTICAL_NLL_TOL),
            'tolerance': PRACTICAL_NLL_TOL,
            'n_paired_folds': len(gap)})

    return pd.DataFrame(rows)

cv_scores_repeated = repeated_cv_scores()

cv_gap_summary = paired_gap_summary(
    cv_scores_repeated,
    comparisons=[
        (MODEL_T5_NAME, MODEL_NAME, 'Does truncating SE history leave performance unchanged?'),
        (HURDLE_AR5_NAME, MODEL_T5_NAME, 'Does flexible Hurdle-AR(5)-S-P match SE-Hurdle-T5?')])

cv_scores_repeated.to_csv(RESULT_DIR / 'repeated_cv_scores.csv', index=False)
cv_gap_summary.to_csv(RESULT_DIR / 'paired_cv_gap_tests.csv', index=False)

In [ ]:
def rmse(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    return float(np.sqrt(np.mean((a[mask] - b[mask]) ** 2))) if mask.any() else np.nan

def ks_d(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    return float(stats.ks_2samp(a, b).statistic) if len(a) and len(b) else np.nan

def gini(values):
    values = np.sort(np.asarray(values, dtype=float).ravel())
    values = values[np.isfinite(values)]
    if len(values) == 0 or values.sum() <= 0:
        return np.nan
    n = len(values)
    cumulative = np.cumsum(values)
    return float((n + 1 - 2 * cumulative.sum() / cumulative[-1]) / n)

def top_share(values, share=0.01):
    values = np.asarray(values, dtype=float).ravel()
    values = values[np.isfinite(values)]
    total = values.sum()
    if len(values) == 0 or total <= 0:
        return np.nan
    k = max(1, int(np.ceil(share * len(values))))
    return float(np.sort(values)[-k:].sum() / total)

def transition_rates(q):
    dropout = []
    restart = []

    for t in range(q.shape[1] - 1):
        prev = q[:, t]
        nxt = q[:, t + 1]
        active = prev > 0
        inactive = prev == 0
        dropout.append(float((nxt[active] == 0).mean()) if active.any() else np.nan)
        restart.append(float((nxt[inactive] > 0).mean()) if inactive.any() else np.nan)

    return np.asarray(dropout), np.asarray(restart)

def rank_curve_for_gap(q):
    ranks = np.column_stack([
        pd.Series(q[:, t]).rank(method='average', pct=True).to_numpy()
        for t in range(q.shape[1])])

    return np.asarray([
        stats.spearmanr(ranks[:, 0], ranks[:, t]).statistic
        for t in range(q.shape[1])], dtype=float)

def model_distances_to_empirical(q, empirical=Q_EMP):
    emp_cum = empirical.sum(axis=1)
    sim_cum = q.sum(axis=1)
    emp_dropout, emp_restart = transition_rates(empirical)
    sim_dropout, sim_restart = transition_rates(q)
    emp_rank = rank_curve_for_gap(empirical)
    sim_rank = rank_curve_for_gap(q)

    return {
        'annual_ks_mean': np.mean([ks_d(empirical[:, t], q[:, t]) for t in range(empirical.shape[1])]),
        'cum_total_ks': ks_d(emp_cum, sim_cum),
        'cum_y5_ks': ks_d(empirical[:, :6].sum(axis=1), q[:, :6].sum(axis=1)),
        'zero_years_ks': ks_d((empirical == 0).sum(axis=1), (q == 0).sum(axis=1)),
        'terminal_rank_abs_error': abs(emp_rank[-1] - sim_rank[-1]),
        'rank_curve_rmse': rmse(emp_rank, sim_rank),
        'dropout_rmse': rmse(emp_dropout, sim_dropout),
        'restart_rmse': rmse(emp_restart, sim_restart),
        'increment_ks': ks_d(np.diff(empirical, axis=1).ravel(), np.diff(q, axis=1).ravel()),
        'gini_abs_error': abs(gini(emp_cum) - gini(sim_cum)),
        'top1_abs_error': abs(top_share(emp_cum, 0.01) - top_share(sim_cum, 0.01))}

diagnostic_simulators = {
    MODEL_NAME: lambda seed: simulate_self_exciting(self_exciting_model, RHO_HAT, len(Q_EMP), seed),
    MODEL_T5_NAME: lambda seed: simulate_self_exciting(self_exciting_t5_model, RHO_T5_HAT, len(Q_EMP), seed, max_lag=TRUNCATED_HISTORY_LENGTH),
    HURDLE_AR5_NAME: lambda seed: simulate_hurdle_ar(hurdle_ar5_model, len(Q_EMP), seed, order=AR_ORDER)}

diagnostic_rows = []

for rep in range(N_DIAGNOSTIC_REPS):
    for j, (label, simulator) in enumerate(diagnostic_simulators.items()):
        q_sim = simulator(SEED + 10000 + 1000 * j + rep)
        row = {'rep': rep, 'model': label}
        row.update(model_distances_to_empirical(q_sim))
        diagnostic_rows.append(row)

diagnostic_replicates = pd.DataFrame(diagnostic_rows)

def diagnostic_gap_summary(diagnostic_replicates, comparisons):
    metrics = [c for c in diagnostic_replicates.columns if c not in {'rep', 'model'}]
    pivot = diagnostic_replicates.pivot_table(index='rep', columns='model', values=metrics)
    rows = []

    for left, right, hypothesis in comparisons:
        for metric in metrics:
            gap = pivot[(metric, left)] - pivot[(metric, right)]
            lo, med, hi = bootstrap_ci(gap, seed=SEED + hash((left, right, metric)) % 100000)
            rows.append({
                'hypothesis': hypothesis,
                'metric': metric,
                'gap': f'{left} minus {right}',
                'mean_gap': gap.mean(),
                'median_gap': gap.median(),
                'ci95_low': lo,
                'ci95_median': med,
                'ci95_high': hi,
                'pct_gap_positive': np.mean(gap > 0),
                'interpretation': 'left worse' if lo > 0 else ('left better' if hi < 0 else 'ambiguous')})
    return pd.DataFrame(rows)

diagnostic_gap_tests = diagnostic_gap_summary(
    diagnostic_replicates,
    comparisons=[(MODEL_T5_NAME, MODEL_NAME, 'Does truncating SE history leave diagnostics unchanged?'),(HURDLE_AR5_NAME, MODEL_T5_NAME, 'Does flexible Hurdle-AR(5)-S-P match SE-Hurdle-T5?')])

diagnostic_replicates.to_csv(RESULT_DIR / 'diagnostic_replicates.csv', index=False)
diagnostic_gap_tests.to_csv(RESULT_DIR / 'diagnostic_gap_tests.csv', index=False)

In [ ]:
rho = 0.6533333333333333
lag = np.arange(1, 6)

se_weights = rho**lag
ar_stage0 = np.array([0.5205, 0, 0, 0, 0])
ar_stage1_4 = np.array([0.4068, 0.1765, 0.0442, -0.0111, -0.0286])
ar_stage5_7 = np.array([0.3729, 0.1860, 0.0703, 0.0912, 0.0433])
ar_stage8_20 = np.array([0.4477, 0.1929, 0.1126, 0.0797, 0.0485])

diagnostic_ar_values = np.vstack([ar_stage1_4,ar_stage5_7, ar_stage8_20])

diagnostic_stage_weights = np.array([4, 3, 13], dtype=float)
diagnostic_stage_weights /= diagnostic_stage_weights.sum()

ar_weighted_mean = np.average(diagnostic_ar_values,axis=0,weights=diagnostic_stage_weights,)

ar_min = diagnostic_ar_values.min(axis=0)
ar_max = diagnostic_ar_values.max(axis=0)

def exponential_profile(k, lag1_weight, decay):
    return lag1_weight * decay ** (k - 1)

(lag1_ar, rho_ar), _ = curve_fit(exponential_profile,lag,ar_weighted_mean,p0=(ar_weighted_mean[0], rho),bounds=([0, 0], [np.inf, 1]))

ar_exponential_fit = exponential_profile(lag,lag1_ar,rho_ar)

total_ss = np.sum((ar_weighted_mean - ar_weighted_mean.mean()) ** 2)

exp_residual_ss = np.sum((ar_weighted_mean - ar_exponential_fit) ** 2)

exp_rmse = np.sqrt(np.mean((ar_weighted_mean - ar_exponential_fit) ** 2))

exp_r2 = 1 - exp_residual_ss / total_ss

fixed_rho_basis = rho ** (lag - 1)

fixed_rho_lag1 = (np.dot(ar_weighted_mean, fixed_rho_basis)/ np.dot(fixed_rho_basis, fixed_rho_basis))

ar_fixed_rho_fit = fixed_rho_lag1 * fixed_rho_basis

fixed_rho_residual_ss = np.sum((ar_weighted_mean - ar_fixed_rho_fit) ** 2)

fixed_rho_rmse = np.sqrt(np.mean((ar_weighted_mean - ar_fixed_rho_fit) ** 2))

fixed_rho_r2 = 1 - fixed_rho_residual_ss / total_ss
stage_colors = [ "#D55E00","#009E73", "#CC79A7","#E69F00"]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.fill_between(lag,ar_min,ar_max,color="0.75",alpha=0.35,zorder=1)

ax.scatter(lag[0],ar_stage0[0],color=stage_colors[0],marker="o",s=38,zorder=4)

for values, color in zip(diagnostic_ar_values,stage_colors[1:]):
    ax.scatter(lag,values,color=color,marker="o",s=38,zorder=4)

ax.plot(lag, ar_weighted_mean, color="black",linewidth=1.6,zorder=3)

ax.scatter(lag,ar_weighted_mean,color="black",marker="o",s=52,zorder=5)

ax.plot(lag,ar_exponential_fit,color="0.25",linestyle="--",linewidth=1.5,zorder=3)
ax.plot(lag,se_weights,color="#0072B2",linewidth=1.8,zorder=3)

ax.scatter(lag, se_weights,color="#0072B2", marker="D",s=54,zorder=5)

ax.set_xticks(lag)
ax.set_xlabel("Lag")
ax.set_ylabel("Coefficient / geometric weight")
ax.set_title("SE geometric decay and fitted Hurdle-AR(5) coefficients")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)


diagnostic_text = (rf"Best AR exponential: $\hat{{\rho}}_{{AR}}={rho_ar:.3f}$"f"\nRMSE = {exp_rmse:.3f}; $R^2$ = {exp_r2:.3f}")

ax.text(0.035,0.055,diagnostic_text,transform=ax.transAxes,ha="left",va="bottom",fontsize=8.5,bbox=dict(boxstyle="round,pad=0.35",facecolor="white",edgecolor="0.65",alpha=0.95))

architecture_handles = [
    Line2D([0], [0], marker="o",linestyle="-",color="black", markerfacecolor="black",markersize=6,label="Hurdle-AR coefficients",),
    Line2D([0], [0], marker="D",linestyle="-",color="#0072B2",markerfacecolor="#0072B2",markersize=6,label="SE geometric weights",)]

architecture_legend = ax.legend(handles=architecture_handles,loc="lower center",bbox_to_anchor=(0.5, 0.92),ncol=2,frameon=False)

ax.add_artist(architecture_legend)

stage_handles = [
    Line2D([0], [0],marker="o",linestyle="none",color=stage_colors[0],label="AR stage 0: lag 1 only"),
    Line2D([0], [0],marker="o",linestyle="none",color=stage_colors[1],label="AR stage 1-4"),
    Line2D([0], [0],marker="o",linestyle="none", color=stage_colors[2],label="AR stage 5-7"),
    Line2D([0],[0],marker="o",linestyle="none",color=stage_colors[3],label="AR stage 8-20"),
    Line2D([0], [0],marker="o",linestyle="-",color="black",label="Duration-weighted AR mean"),
    Line2D([0],[0],linestyle="--",color="0.25",label="Best exponential fit"),
    Patch(facecolor="0.75",alpha=0.35,label="Min/max")]

ax.legend(handles=stage_handles,loc="upper right",
    bbox_to_anchor=(0.98, 0.87),
    ncol=1,fontsize=8,markerscale=0.8,
    labelspacing=0.35,handlelength=1.6,borderpad=0.35,
    frameon=True,framealpha=0.95)

fig.tight_layout()
plt.show()
